# MOSAIC Email Detector
**Detection-only.** Read-only against source data. Every finding requires steward review before any action.

| Cell | What it does | Procedure §§ |
|---|---|---|
| 1 – Overview | This document | — |
| 2 – Rule Reference | Full rule definitions, standardization options, decision workflow | All |
| 3 – Config | Parameters / widgets | — |
| 4 – Constants | Tags, severity weights, regex patterns, standardization rules | All |
| 5 – Discovery | Table & column inventory from information_schema | — |
| 6 – Profiler | Sample-based stats: null rate, format validity, case, whitespace, role accounts, duplicates | §Normalization, §Validation |
| 7 – AI Classifier | Confirm ambiguous candidates; classify semantic role, PII risk, role-account type | §Validation, §Privacy, §Duplicate |
| 8 – Rule Engine | One function per rule group | All |
| 9 – Write Results | Append findings to results Delta table | — |
| 10 – Summary | Blocking findings first, then ranked tables & columns | §Validation, §Enforcement |

> **Hard constraint:** `UPDATE`, `MERGE`, `DELETE`, `CTAS` on source tables are never executed.
> **Scope:** All STRING / VARCHAR / CHAR columns are screened if they look like email fields by name or sample content.
> **AI use:** `ai_query()` is used for (1) ambiguous column confirmation, (2) semantic role classification (member contact, system account, staff, shared inbox), and (3) PII risk tier assignment. All AI decisions are tagged with confidence and require steward review before action.


# MOSAIC Email Detector — Rule Reference

**Detection-only.** No source data is modified.

---

## §Normalization
| Rule tag | What it detects | Required action |
|---|---|---|
| `WHITESPACE_NOT_TRIMMED` | Leading or trailing whitespace in email values | Trim in ETL before Silver; never store raw padded values in curated layers |
| `NOT_LOWERCASE` | Uppercase characters present (domain or local part) | Lowercase in Silver transform unless documented case-sensitive exception |
| `DISPLAY_VARIANT_PRESENT` | Un-normalized display form alongside a normalized column | Remove display variant from Gold unless dictionary-justified |

## §Validation
| Rule tag | What it detects | Required action |
|---|---|---|
| `INVALID_EMAIL_FORMAT` | Values that fail RFC 5322 regex — malformed, garbage (`test`, `none`, placeholder) | NULL or quarantine; do not load into certified dimensions without steward flag |
| `HIGH_INVALID_RATE` | > threshold % of non-null values are invalid | Escalate to steward; block certification until validation is enforced in pipeline |
| `NULL_RATE_HIGH` | > threshold % of values are NULL | Steward reviews; may indicate upstream data loss or wrong column mapping |
| `PATTERN_UNPINNED` | No pattern version recorded in run metadata | Record RFC 5322 regex pattern version before promoting to certified tier |

## §Duplicate and role accounts
| Rule tag | What it detects | Required action |
|---|---|---|
| `ROLE_ACCOUNT_PRESENT` | Shared-inbox prefixes (`info@`, `admin@`, `noreply@`, etc.) | Document in domain rules; do not use as unique person key without product owner approval |
| `DUPLICATE_EMAIL_IN_COLUMN` | Same normalized email appears in > 1 row | Investigate; flag for deduplication or shared-account review |

## §Privacy and security
| Rule tag | What it detects | Required action |
|---|---|---|
| `PII_ENV_RISK` | Email column present in any environment — email is PII and requires documented access controls and masking policy | Verify access controls, masking, and compliance obligations are documented and enforced |
| `PLAIN_EMAIL_IN_LOG_RISK` | Email column name found in a table/schema suggesting log storage | Do not log full email at INFO level; review log schema |

## §Monitoring
| Rule tag | What it detects | Required action |
|---|---|---|
| `VALIDATION_FAILURE_RATE_TREND` | Invalid rate exceeds monthly review threshold | Steward reviews top failure patterns; track trend across runs |

---

## Decision workflow
1. Findings land in the results table with `needs_steward_review = true` where no single automatic action is safe.
2. Steward sets `decided_action` and `decided_by` on each finding.
3. Engineer implements the decided action in the Silver / Gold transform.
4. Re-run detector after fix; confirm finding is resolved.

## Enforcement
Loading unvalidated emails into certified member contact tables is a **certification blocker** until validation is enforced in pipeline.


In [0]:
import re, json, uuid
from datetime import datetime
from typing import Any, Dict, List, Optional, Tuple, Set
from collections import defaultdict
from pyspark.sql import functions as F, DataFrame

def _w(name: str, default) -> str:
    try:
        dbutils.widgets.text(name, str(default))
        return dbutils.widgets.get(name)
    except Exception:
        return str(default)

CFG: Dict[str, Any] = {
    # Source
    "catalog":      _w("catalog",      "data_classification_source"),
    "schema":       _w("schema",       "email_addresses_detector"),
    "table_filter": _w("table_filter", ""),
    "skip_views":   _w("skip_views",   "true").strip().lower() == "true",

    # Layer -- recorded on findings for lineage context
    "layer":        _w("layer",        "Unknown"),   # Bronze | Silver | Gold | Unknown

    # Sampling
    "sample_size":  int(_w("sample_size", 10_000)),
    "seed":         int(_w("seed",        42)),

    # Detection thresholds
    # invalid_rate_threshold: % of non-null values that are invalid before HIGH_INVALID_RATE fires
    "invalid_rate_threshold": float(_w("invalid_rate_threshold", "5.0")),
    # null_rate_threshold: % of values that are NULL before NULL_RATE_HIGH fires
    "null_rate_threshold":    float(_w("null_rate_threshold",    "30.0")),
    # dup_threshold: % of non-null valid emails that are duplicated before DUPLICATE_EMAIL_IN_COLUMN fires
    "dup_threshold":          float(_w("dup_threshold",          "10.0")),
    # string_email_threshold: % of sample that must look like emails for a name-matched column to be confirmed
    "string_email_threshold": float(_w("string_email_threshold", "60.0")),

    # Results
    "out_catalog": _w("out_catalog", "data_classification_target"),
    "out_schema":  _w("out_schema",  "data_classification_output"),
    "out_table":   _w("out_table",   "email_findings"),

    # AI
    "ai_model":    _w("ai_model",   "databricks-meta-llama-3-3-70b-instruct"),
    # enable_ai: when True, uses ai_query() for:
    #   (1) ambiguous column confirmation (is this really an email column?)
    #   (2) semantic role classification (member_contact, staff_contact, shared_inbox, etc.)
    #   (3) PII risk tier assignment
    "enable_ai":   _w("enable_ai",  "true").strip().lower() == "true",

    # RFC 5322 pattern version -- pin this in pipeline docs per §Validation
    "pattern_version": _w("pattern_version", "rfc5322-simplified-v1"),
}

RUN_ID = str(uuid.uuid4())
RUN_TS = datetime.utcnow().isoformat()

print(f"Run      : {RUN_ID}")
print(f"Scope    : {CFG['catalog']}.{CFG['schema']}")
print(f"Layer    : {CFG['layer']}")
print(f"AI       : {'on -> ' + CFG['ai_model'] if CFG['enable_ai'] else 'off (heuristic-only)'}")
print(f"Pattern  : {CFG['pattern_version']}")
print(f"Results  : {CFG['out_catalog']}.{CFG['out_schema']}.{CFG['out_table']}")


In [0]:
# ---------------------------------------------------------------------------
# HOW TO ADD A NEW RULE:
#   1. Add tag to TAGS with procedure section reference
#   2. Add severity to SEVERITY
#   3. Add standardization options to STANDARDIZATION_RULES
#   4. Write _check_<topic>() in Cell 6 and call it in the main loop
# ---------------------------------------------------------------------------

TAGS = {
    # §Normalization
    "WHITESPACE_NOT_TRIMMED":        "§Normalization",
    "NOT_LOWERCASE":                 "§Normalization",
    "DISPLAY_VARIANT_PRESENT":       "§Normalization",
    # §Validation
    "INVALID_EMAIL_FORMAT":          "§Validation",
    "HIGH_INVALID_RATE":             "§Validation",
    "NULL_RATE_HIGH":                "§Validation",
    "PATTERN_UNPINNED":              "§Validation",
    # §Duplicate and role accounts
    "ROLE_ACCOUNT_PRESENT":          "§Duplicate-role-accounts",
    "DUPLICATE_EMAIL_IN_COLUMN":     "§Duplicate-role-accounts",
    # §Privacy and security
    "PII_ENV_RISK":                  "§Privacy",
    "PLAIN_EMAIL_IN_LOG_RISK":       "§Privacy",
    # §Monitoring
    "VALIDATION_FAILURE_RATE_TREND": "§Monitoring",
}

SEVERITY: Dict[str, int] = {
    "INVALID_EMAIL_FORMAT":          10,
    "HIGH_INVALID_RATE":             10,
    "PII_ENV_RISK":                   9,
    "PLAIN_EMAIL_IN_LOG_RISK":        8,
    "NOT_LOWERCASE":                  7,
    "WHITESPACE_NOT_TRIMMED":         7,
    "DUPLICATE_EMAIL_IN_COLUMN":      6,
    "NULL_RATE_HIGH":                 5,
    "ROLE_ACCOUNT_PRESENT":           5,
    "VALIDATION_FAILURE_RATE_TREND":  4,
    "DISPLAY_VARIANT_PRESENT":        3,
    "PATTERN_UNPINNED":               2,
}

# Email column name heuristics
# RE_EMAIL_NAME: column names that strongly suggest an email address field
RE_EMAIL_NAME = re.compile(
    r"(email|e_mail|e-mail|correo|mail_addr|emailaddr|email_address|"
    r"contact_email|member_email|patient_email|user_email|sender|recipient|"
    r"from_addr|to_addr|reply_to|notif_email|alert_email)",
    re.I
)
# RE_EMAIL_NAME_EXCLUDE: looks like email column but stores templates/bodies/flags
RE_EMAIL_NAME_EXCLUDE = re.compile(
    r"(_template$|_body$|_content$|_subject$|_text$|_html$|_message$|"
    r"_type$|_flag$|_status$|_count$|_ind$|_indicator$)",
    re.I
)

# Semantic roles -- the purpose / business use of the email column
SEMANTIC_ROLES = {
    "member_contact", "patient_contact", "staff_contact",
    "system_account", "shared_inbox", "notification_target", "other"
}

# Role-account prefixes per §Duplicate: document in domain rules
ROLE_PREFIXES = [
    "info", "admin", "administrator", "support", "help", "helpdesk",
    "noreply", "no-reply", "donotreply", "do-not-reply",
    "postmaster", "webmaster", "hostmaster", "abuse",
    "billing", "sales", "marketing", "ops", "operations",
    "contact", "hello", "team", "office", "reception",
    "careers", "jobs", "hr", "it", "security", "privacy",
    "legal", "compliance", "finance", "accounting",
    "newsletter", "notifications", "alerts", "updates",
    "bot", "daemon", "mailer", "bounce",
]
ROLE_PREFIXES_SET = set(ROLE_PREFIXES)
ROLE_PREFIXES_SQL  = ", ".join(f"'{p}'" for p in ROLE_PREFIXES)

# Log table name heuristics -- for PLAIN_EMAIL_IN_LOG_RISK
RE_LOG_TABLE = re.compile(
    r"(\blog\b|\blogs\b|audit|audit_trail|event_log|activity_log|"
    r"access_log|app_log|trace|traces|journal|changelog)",
    re.I
)

# RFC 5322 simplified regex (pinned via CFG["pattern_version"])
# Source: emailregex.com -- simplified production-safe variant covering 99.9%+ of real addresses.
# Pin version in pipeline docs per §Validation.
EMAIL_REGEX_PY  = re.compile(r"^[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}$")
EMAIL_REGEX_SQL = r"^[a-zA-Z0-9._%+\\-]+@[a-zA-Z0-9.\\-]+\\.[a-zA-Z]{2,}$"

# Garbage / placeholder values that must not reach certified dimensions
GARBAGE_VALUES = {
    "test", "test@test.com", "test@example.com", "none", "null",
    "n/a", "na", "noemail", "no_email", "noemail@noemail.com",
    "unknown", "unknown@unknown.com", "placeholder", "example@example.com",
    "user@example.com", "email@email.com", "fake@fake.com",
    "donotreply@donotreply.com", "invalid", "invalid@invalid.com",
    "notset", "not_set", "empty", "blank", "tbd",
}
GARBAGE_SQL = ", ".join(f"'{g}'" for g in GARBAGE_VALUES)

STANDARDIZATION_RULES: Dict[str, list] = {

    "WHITESPACE_NOT_TRIMMED": [
        {"option_key": "trim_in_silver",
         "label":      "Trim leading/trailing whitespace in Silver transform",
         "sql":        "TRIM(email_col) AS email_col",
         "notes":      "Never store padded values in curated layers (§Normalization). "
                       "Apply TRIM() before any other normalization step."},
    ],

    "NOT_LOWERCASE": [
        {"option_key": "lowercase_in_silver",
         "label":      "Lowercase email in Silver transform",
         "sql":        "LOWER(TRIM(email_col)) AS email_col",
         "notes":      "Store in lowercase in curated tables (§Normalization). "
                       "Exception: case-sensitive external system of record must be documented."},
        {"option_key": "document_exception",
         "label":      "Document case-sensitive exception in data dictionary",
         "sql":        "-- No transform. Document exception in data dictionary.",
         "notes":      "Use only when source system is a documented case-sensitive system of record."},
    ],

    "DISPLAY_VARIANT_PRESENT": [
        {"option_key": "remove_display_variant",
         "label":      "Remove un-normalized display variant from Gold",
         "sql":        "-- Drop display variant column from Gold model; retain only normalized column.",
         "notes":      "Do not persist display-only variants alongside normalized column in Gold "
                       "without dictionary justification (§Normalization)."},
        {"option_key": "document_justification",
         "label":      "Document business justification in data dictionary",
         "sql":        "-- Retain display variant. Add dictionary entry with justification.",
         "notes":      "Use when downstream consumer requires original casing for display only."},
    ],

    "INVALID_EMAIL_FORMAT": [
        {"option_key": "null_invalid",
         "label":      "Set invalid emails to NULL in Silver transform",
         "sql":        "CASE WHEN LOWER(TRIM(email_col)) RLIKE '<email_regex>' "
                       "THEN LOWER(TRIM(email_col)) ELSE NULL END AS email_col",
         "notes":      "Invalid addresses -> NULL per §Validation. "
                       "Do not load garbage into certified dimensions without steward flag. "
                       "Pin pattern version in pipeline docs: see pattern_version field."},
        {"option_key": "quarantine_invalid",
         "label":      "Route invalid rows to quarantine table",
         "sql":        "-- Route rows WHERE NOT (LOWER(TRIM(email_col)) RLIKE '<email_regex>') "
                       "to quarantine table.",
         "notes":      "Use when steward needs to review invalid values before nulling. "
                       "Invalid records must not reach certified dimensions."},
    ],

    "HIGH_INVALID_RATE": [
        {"option_key": "quarantine_and_escalate",
         "label":      "Quarantine all invalid rows and escalate to steward",
         "sql":        "-- Route all non-matching rows to quarantine. Block certification.",
         "notes":      "Loading unvalidated emails into certified member contact tables "
                       "is a certification blocker (§Enforcement). "
                       "Steward reviews top failure patterns monthly (§Monitoring)."},
    ],

    "NULL_RATE_HIGH": [
        {"option_key": "investigate_source",
         "label":      "Investigate source for upstream data loss or wrong column mapping",
         "sql":        "-- No transform. Investigate source system and pipeline mapping.",
         "notes":      "High null rate may indicate upstream data loss, wrong join key, "
                       "or misidentified email column."},
    ],

    "PATTERN_UNPINNED": [
        {"option_key": "pin_pattern_version",
         "label":      "Record RFC 5322 pattern version in pipeline docs and data dictionary",
         "sql":        "-- Set pattern_version widget and commit to pipeline repo.",
         "notes":      "§Validation requires pattern version to be recorded in repo. "
                       "Reference: emailregex.com -- pin chosen pattern in pipeline docs."},
    ],

    "ROLE_ACCOUNT_PRESENT": [
        {"option_key": "document_domain_rules",
         "label":      "Document shared-inbox handling in domain rules",
         "sql":        "-- Add domain rule: shared inboxes must not be used as unique person keys.",
         "notes":      "Do not use role accounts as unique person keys without product owner approval "
                       "(§Duplicate). Document all role-account prefixes in the data dictionary."},
        {"option_key": "exclude_from_person_key",
         "label":      "Exclude role accounts from person-key and deduplication logic",
         "sql":        "-- Filter: WHERE NOT (SPLIT_PART(LOWER(email_col), '@', 1) "
                       "IN (<role_prefix_list>))",
         "notes":      "Use when building member/patient unique keys or EMPI matching inputs."},
    ],

    "DUPLICATE_EMAIL_IN_COLUMN": [
        {"option_key": "investigate_shared_account",
         "label":      "Investigate whether duplicates are shared accounts or data errors",
         "sql":        "-- SELECT LOWER(TRIM(email_col)), COUNT(*) FROM tbl "
                       "GROUP BY 1 HAVING COUNT(*) > 1 ORDER BY 2 DESC",
         "notes":      "Duplicates may be shared inboxes, data entry errors, or multi-member households. "
                       "Steward decides per case (§Duplicate)."},
        {"option_key": "deduplicate_downstream",
         "label":      "Apply deduplication in Gold model",
         "sql":        "-- ROW_NUMBER() OVER ("
                       "PARTITION BY LOWER(TRIM(email_col)) ORDER BY <priority_col>) = 1",
         "notes":      "Use when duplicates are confirmed errors, not shared accounts."},
    ],

    "PII_ENV_RISK": [
        {"option_key": "mask_in_lower_env",
         "label":      "Apply PII masking or synthetic replacement in non-production",
         "sql":        "-- CASE WHEN env != 'prod' THEN SHA2(email_col, 256) ELSE email_col END",
         "notes":      "Email is PII (§Privacy). "
                       "Verify access controls, masking policy, and compliance obligations "
                       "are documented and enforced for this column. "
                       "Document any approved exception with an approval record."},
        {"option_key": "document_exception",
         "label":      "Document approved exception for real PII in non-production",
         "sql":        "-- No transform. Add data dictionary entry with approval record.",
         "notes":      "Use only when non-prod PII access is explicitly approved and documented."},
    ],

    "PLAIN_EMAIL_IN_LOG_RISK": [
        {"option_key": "review_log_schema",
         "label":      "Review log table schema; redact email from INFO-level logs",
         "sql":        "-- Do not SELECT email columns into log/audit tables at INFO level. "
                       "-- Use SHA2(email_col, 256) AS email_hash in log tables.",
         "notes":      "Do not log full email in application or job logs at INFO level (§Privacy). "
                       "Use hashed or masked representation in logs."},
    ],

    "VALIDATION_FAILURE_RATE_TREND": [
        {"option_key": "steward_monthly_review",
         "label":      "Flag for steward monthly review of top failure patterns",
         "sql":        "-- SELECT email_col, COUNT(*) FROM quarantine "
                       "GROUP BY 1 ORDER BY 2 DESC LIMIT 20",
         "notes":      "Track validation failure rate; steward reviews top failure patterns "
                       "monthly during source onboarding (§Monitoring)."},
    ],
}

print(f"Constants loaded: {len(TAGS)} tags | {len(STANDARDIZATION_RULES)} standardization entries")
print(f"Role prefixes   : {len(ROLE_PREFIXES_SET)}")
print(f"Garbage values  : {len(GARBAGE_VALUES)}")
print(f"Pattern version : {CFG['pattern_version']}")


In [0]:
# ---------------------------------------------------------------------------
# Reads Unity Catalog information_schema.
# Builds: tables, table_cols
# Only STRING / VARCHAR / CHAR columns are candidates for email detection.
# ---------------------------------------------------------------------------

cat, sch = CFG["catalog"], CFG["schema"]

def _esc(name: str) -> str:
    return name.replace("`", "``")

STRING_DTYPES_PFX = ("STRING", "VARCHAR", "CHAR")

view_clause = "AND table_type IN ('MANAGED', 'EXTERNAL')" if CFG["skip_views"] else ""
tables: List[str] = [
    r.table_name for r in spark.sql(f"""
        SELECT table_name FROM `{_esc(cat)}`.information_schema.tables
        WHERE  table_schema = '{_esc(sch)}' {view_clause}
        ORDER BY table_name
    """).collect()
]

if CFG["table_filter"].strip():
    pat = re.compile(CFG["table_filter"], re.I)
    tables = [t for t in tables if pat.search(t)]

if not tables:
    raise ValueError(f"No tables found in {cat}.{sch} -- check catalog/schema/table_filter.")

tbl_in = ", ".join(f"'{_esc(t)}'" for t in tables)
table_cols: Dict[str, List[Tuple[str, str]]] = defaultdict(list)
for r in spark.sql(f"""
    SELECT table_name, column_name, data_type
    FROM   `{_esc(cat)}`.information_schema.columns
    WHERE  table_schema = '{_esc(sch)}' AND table_name IN ({tbl_in})
    ORDER BY table_name, ordinal_position
""").collect():
    dt = r.data_type.upper()
    if any(dt.startswith(p) for p in STRING_DTYPES_PFX):
        table_cols[r.table_name].append((r.column_name, dt))

total_string_cols = sum(len(v) for v in table_cols.values())
print(f"Scope        : {cat}.{sch}")
print(f"Tables       : {len(tables)}")
print(f"String cols  : {total_string_cols}")
print(f"Layer        : {CFG['layer']}")


In [0]:
# ---------------------------------------------------------------------------
# PROFILER -- key design decisions:
#
# 1. screen_email_col(): sample-based name + content heuristic.
#    Tries EMAIL_REGEX_PY on up to 2000 sampled values per column.
#
# 2. profile_table_batch(): ONE SQL per table for ALL aggregate stats:
#    null count, invalid count (via RLIKE), whitespace count, uppercase
#    count, role-account count, garbage count.
#
# 3. Duplicate count: one SQL per column (requires GROUP BY / HAVING);
#    runs only for confirmed candidates.
#
# 4. Sample invalid values collected from cached sample DF for steward context.
#
# 5. sample_df cached per table; unpersisted after the table loop.
# ---------------------------------------------------------------------------

def _esc(name: str) -> str:
    return name.replace("`", "``")

def get_sample(tbl: str) -> DataFrame:
    fq = f"`{_esc(cat)}`.`{_esc(sch)}`.`{_esc(tbl)}`"
    n  = CFG["sample_size"]
    try:
        return spark.sql(f"SELECT * FROM {fq} TABLESAMPLE ({n} ROWS)")
    except Exception:
        total = spark.sql(f"SELECT COUNT(*) AS n FROM {fq}").collect()[0]["n"]
        return (
            spark.table(f"`{_esc(cat)}`.`{_esc(sch)}`.`{_esc(tbl)}`")
            .sample(withReplacement=False, fraction=min(1.0, n / max(1, total)), seed=CFG["seed"])
            .limit(n)
        )


def screen_email_col(col: str, sample_df: DataFrame) -> Tuple[bool, float]:
    """
    Estimate what fraction of non-null values look like valid email addresses.
    Uses EMAIL_REGEX_PY on the CACHED sample DF -- no full table scan.
    Returns (is_candidate, email_rate_pct).
    """
    name_match = bool(RE_EMAIL_NAME.search(col)) and not bool(RE_EMAIL_NAME_EXCLUDE.search(col))
    try:
        rows = (
            sample_df
            .select(F.col(f"`{_esc(col)}`").alias("v"))
            .filter(F.col("v").isNotNull() & (F.trim(F.col("v")) != ""))
            .limit(2000)
            .collect()
        )
        total   = len(rows)
        if total == 0:
            return name_match, 0.0
        matched = sum(1 for r in rows if EMAIL_REGEX_PY.match(str(r["v"]).strip().lower()))
        rate    = matched / total * 100
        is_cand = name_match or rate >= CFG["string_email_threshold"]
        return is_cand, rate
    except Exception:
        return name_match, 0.0


def profile_table_batch(tbl: str, candidates: Dict[str, dict]) -> Dict[str, dict]:
    """
    ONE SQL per table collecting ALL aggregate stats for ALL candidate columns.
    Stats per column: null_count, invalid_count, ws_count, upper_count,
    role_count, garbage_count. Duplicate count computed separately (needs GROUP BY).
    """
    if not candidates:
        return candidates

    fq    = f"`{_esc(cat)}`.`{_esc(sch)}`.`{_esc(tbl)}`"
    exprs = ["COUNT(*) AS __total__"]

    for col, info in candidates.items():
        cs   = f"`{_esc(col)}`"
        safe = col.replace("`","").replace(" ","_").replace(".","_")[:60]
        norm = f"LOWER(TRIM({cs}))"

        exprs += [
            f"COUNT(*) - COUNT({cs}) AS `__null__{safe}`",
            f"COUNT(CASE WHEN {cs} IS NOT NULL AND TRIM({cs}) != '' "
            f"  AND NOT ({norm} RLIKE '{EMAIL_REGEX_SQL}') THEN 1 END) AS `__invalid__{safe}`",
            f"COUNT(CASE WHEN {cs} IS NOT NULL AND {cs} != TRIM({cs}) THEN 1 END) AS `__ws__{safe}`",
            f"COUNT(CASE WHEN {cs} IS NOT NULL AND {norm} RLIKE '{EMAIL_REGEX_SQL}' "
            f"  AND {cs} != {norm} THEN 1 END) AS `__upper__{safe}`",
            f"COUNT(CASE WHEN {cs} IS NOT NULL AND {norm} RLIKE '{EMAIL_REGEX_SQL}' "
            f"  AND SPLIT_PART({norm}, '@', 1) IN ({ROLE_PREFIXES_SQL}) "
            f"  THEN 1 END) AS `__role__{safe}`",
            f"COUNT(CASE WHEN LOWER(TRIM({cs})) IN ({GARBAGE_SQL}) "
            f"  THEN 1 END) AS `__garbage__{safe}`",
        ]

    try:
        row = spark.sql(f"SELECT {', '.join(exprs)} FROM {fq}").collect()[0].asDict()
    except Exception as e:
        print(f"  [WARN] Batch profile failed for {tbl}: {e}")
        return candidates

    total = row["__total__"] or 0
    for col, info in candidates.items():
        safe = col.replace("`","").replace(" ","_").replace(".","_")[:60]
        info.update({
            "total":         total,
            "null_count":    row.get(f"__null__{safe}", 0) or 0,
            "invalid_count": row.get(f"__invalid__{safe}", 0) or 0,
            "ws_count":      row.get(f"__ws__{safe}", 0) or 0,
            "upper_count":   row.get(f"__upper__{safe}", 0) or 0,
            "role_count":    row.get(f"__role__{safe}", 0) or 0,
            "garbage_count": row.get(f"__garbage__{safe}", 0) or 0,
        })

    # Duplicate count: one SQL per column (GROUP BY requires separate query)
    for col, info in candidates.items():
        cs   = f"`{_esc(col)}`"
        norm = f"LOWER(TRIM({cs}))"
        try:
            dup_count = spark.sql(f"""
                SELECT COUNT(*) AS n FROM (
                    SELECT {norm} AS nv FROM `{_esc(cat)}`.`{_esc(sch)}`.`{_esc(tbl)}`
                    WHERE {cs} IS NOT NULL AND TRIM({cs}) != ''
                      AND {norm} RLIKE '{EMAIL_REGEX_SQL}'
                    GROUP BY nv HAVING COUNT(*) > 1
                ) t
            """).collect()[0]["n"] or 0
            info["dup_email_count"] = dup_count
        except Exception:
            info["dup_email_count"] = 0

    # Sample invalid values from the cached sample DF for steward context
    for col, info in candidates.items():
        cs = f"`{_esc(col)}`"
        try:
            inv_samples = [
                str(r["v"]) for r in
                table_samples.get(tbl, spark.range(0))
                .select(F.col(cs).alias("v"))
                .filter(
                    F.col("v").isNotNull() &
                    (F.trim(F.col("v")) != "") &
                    (~F.lower(F.trim(F.col("v"))).rlike(EMAIL_REGEX_SQL))
                )
                .limit(10)
                .collect()
            ]
        except Exception:
            inv_samples = []
        info["sample_invalids"] = inv_samples

    return candidates


# Run profiler
table_samples:    Dict[str, DataFrame]       = {}
email_candidates: Dict[str, Dict[str, dict]] = defaultdict(dict)
total_screened    = 0
total_candidates  = 0

for tbl in tables:
    cols      = table_cols.get(tbl, [])
    sample_df = get_sample(tbl)
    table_samples[tbl] = sample_df

    for col, dtype in cols:
        total_screened += 1
        if RE_EMAIL_NAME_EXCLUDE.search(col):
            continue

        is_cand, email_rate = screen_email_col(col, sample_df)
        if not is_cand:
            continue

        candidate_reason = (
            "name_heuristic" if RE_EMAIL_NAME.search(col) else "content_rate"
        )

        try:
            sample_vals = [
                r["v"] for r in
                sample_df
                .select(F.col(f"`{_esc(col)}`").cast("string").alias("v"))
                .filter(F.col("v").isNotNull())
                .limit(6)
                .collect()
            ]
        except Exception:
            sample_vals = []

        email_candidates[tbl][col] = {
            "dtype":               dtype,
            "total":               0,
            "null_count":          0,
            "invalid_count":       0,
            "ws_count":            0,
            "upper_count":         0,
            "role_count":          0,
            "garbage_count":       0,
            "dup_email_count":     0,
            "sample_invalids":     [],
            "email_rate":          email_rate,
            "candidate_reason":    candidate_reason,
            "needs_ai_confirm":    True,
            "sample_vals":         sample_vals,
            "semantic_role":       "other",
            "semantic_source":     "heuristic",
            "semantic_confidence": "medium",
            "pii_risk":            "high",   # default: all emails are PII
            "is_likely_unique_key":False,
            "confirmed":           True,
        }
        total_candidates += 1

    if email_candidates[tbl]:
        email_candidates[tbl] = profile_table_batch(tbl, email_candidates[tbl])


print(f"Profiler done.")
print(f"  Screened  : {total_screened} string columns across {len(tables)} table(s)")
print(f"  Candidates: {total_candidates} email column(s)")
for tbl, cols in email_candidates.items():
    print(f"  {tbl}: {len(cols)} candidate(s)")


In [0]:
# ---------------------------------------------------------------------------
# AI EMAIL CLASSIFIER -- scalable batch design:
#
# All ai_query() calls are chunked to max BATCH_SIZE columns per call.
#
# Two responsibilities per table:
# 1. COLUMN CONFIRMATION: is this actually an email column? Catches false
#    positives from the name heuristic (e.g. email_template_body).
#    Retention rules mirror the Date Detector: name-matched columns and
#    columns with email_rate >= 50% are kept even on AI rejection, since
#    both are stronger signals than a 6-sample LLM judgment.
# 2. SEMANTIC ROLE + PII TIER: classifies business purpose and privacy risk.
# ---------------------------------------------------------------------------

BATCH_SIZE = 20

def _ai_query(prompt: str) -> str:
    safe = prompt.replace("'", "''")
    raw  = spark.sql(
        f"SELECT ai_query('{CFG['ai_model']}', '{safe}', failOnError => false) AS r"
    ).collect()[0]["r"]
    if hasattr(raw, "errorStatus") and raw.errorStatus:
        raise ValueError(raw.errorStatus)
    return raw.result if hasattr(raw, "result") else str(raw)

def _chunked(items: list, size: int = BATCH_SIZE):
    for i in range(0, len(items), size):
        yield items[i:i + size]


def classify_table(tbl: str) -> None:
    cands = email_candidates.get(tbl, {})
    if not cands:
        return

    if not CFG["enable_ai"]:
        for col in list(cands.keys()):
            info = cands[col]
            if info["candidate_reason"] == "content_rate" and info["email_rate"] < 50.0:
                del email_candidates[tbl][col]
        return

    # Step 1: Column confirmation
    for chunk_items in _chunked(list(cands.items())):
        payload = json.dumps([{
            "col":        col,
            "samples":    info["sample_vals"][:6],
            "email_rate": round(info["email_rate"], 1),
            "reason":     info["candidate_reason"],
        } for col, info in chunk_items], default=str)

        prompt = (
            "Strict data classifier. Reply ONLY with a JSON array -- no prose, no markdown. "
            f"Table: {tbl}. Determine if each STRING column stores email addresses "
            "(not templates, bodies, subject lines, or labels). "
            "Consider column name, sample values, and email parse rate. "
            f"Columns: {payload}. "
            'Return: [{"col":"<n>","is_email":true|false,"confidence":"high|medium|low"}]'
        )
        try:
            for item in json.loads(_ai_query(prompt)):
                col = item.get("col", "")
                if col not in cands:
                    continue
                if not item.get("is_email", False):
                    info = email_candidates[tbl][col]
                    # Keep name-matched or high-rate columns regardless of AI verdict
                    if info["candidate_reason"] == "name_heuristic" or info["email_rate"] >= 50.0:
                        info["ai_flagged_not_email"] = True
                    else:
                        info["confirmed"] = False
                else:
                    email_candidates[tbl][col]["ai_confirm_conf"] = item.get("confidence", "low")
        except Exception as e:
            print(f"  [WARN] AI confirm chunk failed for {tbl}: {e}")
            # Fallback: keep name-heuristic and high-rate columns
            for col, info in chunk_items:
                if info["candidate_reason"] != "name_heuristic" and info["email_rate"] < 50.0:
                    info["confirmed"] = False

    # Remove rejected candidates
    for col in list(email_candidates[tbl].keys()):
        if not email_candidates[tbl][col].get("confirmed", True):
            del email_candidates[tbl][col]

    if not email_candidates[tbl]:
        return

    # Step 2: Semantic role + PII tier
    confirmed = list(email_candidates[tbl].keys())
    for chunk in _chunked(confirmed):
        payload = json.dumps([{
            "col":        col,
            "table":      tbl,
            "samples":    email_candidates[tbl][col]["sample_vals"][:6],
            "role_count": email_candidates[tbl][col]["role_count"],
        } for col in chunk if col in email_candidates[tbl]], default=str)

        prompt = (
            "Data classification expert. Reply ONLY with a JSON array -- no prose, no markdown. "
            f"Table: {tbl}. For each confirmed email column classify: "
            "(1) semantic_role: member_contact (health plan member), patient_contact (clinical patient), "
            "staff_contact (employee/provider/staff), system_account (automated system sender/receiver), "
            "shared_inbox (team or role-based inbox), notification_target (transactional/alert only), other. "
            "(2) pii_risk: high (real person contact email, directly identifiable), "
            "medium (pseudonymous, role account, or indirect), low (system account with no person linkage). "
            "(3) is_likely_unique_key: true if the column appears to serve as a person/record identifier. "
            "(4) confidence: high/medium/low. "
            f"Columns: {payload}. "
            'Return: [{"col":"<n>","semantic_role":"<r>","pii_risk":"<p>",'
            '"is_likely_unique_key":true|false,"confidence":"high|medium|low"}]'
        )
        try:
            for item in json.loads(_ai_query(prompt)):
                col = item.get("col", "")
                if col not in email_candidates[tbl]:
                    continue
                email_candidates[tbl][col].update({
                    "semantic_role":       item.get("semantic_role", "other"),
                    "semantic_source":     "ai",
                    "semantic_confidence": item.get("confidence", "low"),
                    "pii_risk":            item.get("pii_risk", "high"),
                    "is_likely_unique_key":bool(item.get("is_likely_unique_key", False)),
                })
        except Exception as e:
            print(f"  [WARN] AI semantic chunk failed for {tbl}: {e}")
            for col in chunk:
                if col in email_candidates[tbl]:
                    email_candidates[tbl][col]["ai_classification_error"] = str(e)


# Run classifier
for tbl in list(email_candidates.keys()):
    classify_table(tbl)
    remaining = len(email_candidates[tbl])
    print(f"  ok {tbl}: {remaining} confirmed email column(s)")

total_confirmed = sum(len(v) for v in email_candidates.values())
print(f"\nAI classification done. {total_confirmed} confirmed email column(s).")


In [0]:
# ---------------------------------------------------------------------------
# RULE ENGINE:
# All aggregate stats come from the pre-computed dict in email_candidates.
# No additional full-table SQL runs here.
# ---------------------------------------------------------------------------

def _esc(name: str) -> str:
    return name.replace("`", "``")


def _finding(tbl, col, info, tag, count, total, samples, action,
             hint=None, confidence="high",
             std_options=None, auto_decided_action=None) -> dict:
    pct     = round(count / total * 100, 4) if total else 0.0
    options = std_options if std_options is not None else STANDARDIZATION_RULES.get(tag, [])
    decided = (auto_decided_action
               if auto_decided_action is not None and len(options) == 1
               else None)
    return {
        "run_id":                   RUN_ID,
        "run_ts":                   RUN_TS,
        "catalog":                  cat,
        "schema":                   sch,
        "table_name":               tbl,
        "column_name":              col,
        "data_type":                info.get("dtype", "UNKNOWN"),
        "layer":                    CFG["layer"],
        "semantic_role":            info.get("semantic_role", "other"),
        "pii_risk":                 info.get("pii_risk", "high"),
        "is_likely_unique_key":     bool(info.get("is_likely_unique_key", False)),
        "semantic_source":          info.get("semantic_source", "heuristic"),
        "rule_ref":                 TAGS.get(tag, "§?"),
        "classification":           tag,
        "pattern_version":          CFG["pattern_version"],
        "affected_count":           int(count),
        "affected_pct":             float(pct),
        "total_rows":               int(total),
        "sample_values":            json.dumps(samples, default=str),
        "recommended_action":       action,
        "dictionary_strategy_hint": hint,
        "standardization_rule":     json.dumps(options, ensure_ascii=False),
        "confidence":               confidence,
        "needs_steward_review":     decided is None,
        "decided_action":           decided,
        "decided_by":               None,
    }


# §Normalization checks
def _check_normalization(tbl, col, info) -> list:
    total    = info["total"]
    findings = []

    if info["ws_count"] > 0:
        findings.append(_finding(tbl, col, info, "WHITESPACE_NOT_TRIMMED",
            info["ws_count"], total,
            info["sample_vals"][:3],
            f"{info['ws_count']:,} value(s) have leading/trailing whitespace. "
            "Trim in Silver transform before any other normalization (§Normalization).",
            confidence="high",
            auto_decided_action="trim_in_silver",
        ))

    if info["upper_count"] > 0:
        findings.append(_finding(tbl, col, info, "NOT_LOWERCASE",
            info["upper_count"], total,
            info["sample_vals"][:3],
            f"{info['upper_count']:,} valid email value(s) contain uppercase characters. "
            "Store in lowercase in curated tables unless a documented case-sensitive "
            "exception exists (§Normalization).",
            confidence="high",
        ))

    return findings


# §Validation checks
def _check_validation(tbl, col, info) -> list:
    total      = info["total"]
    null_count = info["null_count"]
    non_null   = total - null_count
    findings   = []

    invalid  = info["invalid_count"]
    garbage  = info["garbage_count"]
    total_bad = max(invalid, garbage)

    # Any single bad value must be flagged -- unconditional, not threshold-gated
    if total_bad > 0:
        findings.append(_finding(tbl, col, info, "INVALID_EMAIL_FORMAT",
            total_bad, total,
            info["sample_invalids"][:5],
            f"{total_bad:,} value(s) fail RFC 5322 format validation "
            f"(pattern: {CFG['pattern_version']}). "
            "Set to NULL or route to quarantine; do not load into certified dimensions "
            "without steward flag (§Validation).",
            confidence="high",
        ))

    # High invalid rate -- rate-threshold signal for escalation
    if non_null > 0:
        fail_pct = invalid / non_null * 100
        if fail_pct > CFG["invalid_rate_threshold"]:
            findings.append(_finding(tbl, col, info, "HIGH_INVALID_RATE",
                invalid, total,
                info["sample_invalids"][:5],
                f"{fail_pct:.1f}% of non-null values are invalid. "
                "Loading unvalidated emails into certified member contact tables "
                "is a certification blocker (§Enforcement). "
                "Escalate to steward; track failure patterns monthly (§Monitoring).",
                confidence="high",
                auto_decided_action="quarantine_and_escalate",
            ))

    # High null rate
    if total > 0:
        null_pct = null_count / total * 100
        if null_pct > CFG["null_rate_threshold"]:
            findings.append(_finding(tbl, col, info, "NULL_RATE_HIGH",
                null_count, total, [],
                f"{null_pct:.1f}% of values are NULL. "
                "Investigate source for upstream data loss or wrong column mapping (§Validation).",
                confidence="medium",
            ))

    # Pattern version not pinned
    if not CFG.get("pattern_version", "").strip() or CFG["pattern_version"] in ("", "Unknown"):
        findings.append(_finding(tbl, col, info, "PATTERN_UNPINNED",
            0, total, [],
            "RFC 5322 pattern version not recorded in run metadata. "
            "Pin the pattern version in pipeline docs and data dictionary (§Validation).",
            confidence="high",
            auto_decided_action="pin_pattern_version",
        ))

    return findings


# §Duplicate and role accounts checks
def _check_duplicates_and_roles(tbl, col, info) -> list:
    total    = info["total"]
    findings = []

    if info["role_count"] > 0:
        findings.append(_finding(tbl, col, info, "ROLE_ACCOUNT_PRESENT",
            info["role_count"], total,
            info["sample_vals"][:3],
            f"{info['role_count']:,} value(s) match shared-inbox / role-account prefixes "
            "(e.g. info@, admin@, noreply@). "
            "Document in domain rules; do not use as unique person keys without "
            "product owner approval (§Duplicate).",
            confidence="high",
        ))

    if info["dup_email_count"] > 0:
        valid_non_null = max(1, info["total"] - info["null_count"] - info["invalid_count"])
        dup_pct = info["dup_email_count"] / valid_non_null * 100
        if dup_pct > CFG["dup_threshold"]:
            findings.append(_finding(tbl, col, info, "DUPLICATE_EMAIL_IN_COLUMN",
                info["dup_email_count"], total, [],
                f"{info['dup_email_count']:,} normalized email value(s) appear in more than one row "
                f"({dup_pct:.1f}% of valid non-null emails). "
                "Investigate: shared accounts, data entry errors, or multi-member households (§Duplicate).",
                confidence="medium",
            ))

    return findings


# §Privacy and security checks
def _check_privacy(tbl, col, info) -> list:
    total    = info["total"]
    findings = []

    # PII_ENV_RISK fires in every environment -- email is PII regardless of
    # where the data sits. Access controls, masking, and compliance obligations
    # apply in production just as much as lower environments.
    non_null = total - info["null_count"]
    if non_null > 0:
        findings.append(_finding(tbl, col, info, "PII_ENV_RISK",
            non_null, total, [],
            "Email is PII -- verify access controls, masking policy, and compliance obligations "
            "are documented and enforced for this column (§Privacy).",
            confidence="high",
        ))

    if RE_LOG_TABLE.search(tbl):
        findings.append(_finding(tbl, col, info, "PLAIN_EMAIL_IN_LOG_RISK",
            0, total, [],
            f"Email column '{col}' found in table '{tbl}' which appears to be a log/audit table. "
            "Do not log full email at INFO level (§Privacy). "
            "Use SHA2(email_col, 256) or a masked form in log tables.",
            confidence="medium",
        ))

    return findings


# Main loop
all_findings: List[dict] = []

for tbl, cols in email_candidates.items():
    tbl_count = 0
    for col, info in cols.items():
        col_findings = (
            _check_normalization(tbl, col, info)
            + _check_validation(tbl, col, info)
            + _check_duplicates_and_roles(tbl, col, info)
            + _check_privacy(tbl, col, info)
        )
        all_findings.extend(col_findings)
        tbl_count += len(col_findings)
    print(f"  ok {tbl}: {tbl_count} finding(s)")

print(f"\nRule engine done. {len(all_findings)} total finding(s).")


In [0]:
from pyspark.sql.types import (StructType, StructField, StringType,
                                LongType, DoubleType, BooleanType)

SCHEMA = StructType([
    StructField("run_id",                   StringType(),  True),
    StructField("run_ts",                   StringType(),  True),
    StructField("catalog",                  StringType(),  True),
    StructField("schema",                   StringType(),  True),
    StructField("table_name",               StringType(),  True),
    StructField("column_name",              StringType(),  True),
    StructField("data_type",                StringType(),  True),
    StructField("layer",                    StringType(),  True),
    StructField("semantic_role",            StringType(),  True),
    StructField("pii_risk",                 StringType(),  True),
    StructField("is_likely_unique_key",     BooleanType(), True),
    StructField("semantic_source",          StringType(),  True),
    StructField("rule_ref",                 StringType(),  True),
    StructField("classification",           StringType(),  True),
    StructField("pattern_version",          StringType(),  True),
    StructField("affected_count",           LongType(),    True),
    StructField("affected_pct",             DoubleType(),  True),
    StructField("total_rows",               LongType(),    True),
    StructField("sample_values",            StringType(),  True),
    StructField("recommended_action",       StringType(),  True),
    StructField("dictionary_strategy_hint", StringType(),  True),
    StructField("standardization_rule",     StringType(),  True),
    StructField("confidence",               StringType(),  True),
    StructField("needs_steward_review",     BooleanType(), True),
    StructField("decided_action",           StringType(),  True),
    StructField("decided_by",               StringType(),  True),
])

out_fq  = f"`{CFG['out_catalog']}`.`{CFG['out_schema']}`.`{CFG['out_table']}`"
out_tbl = f"{CFG['out_catalog']}.{CFG['out_schema']}.{CFG['out_table']}"

findings_df = spark.createDataFrame(all_findings or [], schema=SCHEMA)

if all_findings:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CFG['out_catalog']}`.`{CFG['out_schema']}`")
    findings_df.write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(out_tbl)
    print(f"ok  {len(all_findings):,} finding(s) written to {out_fq}")
else:
    print("No findings generated.")

print(f"Run ID: {RUN_ID}")


In [0]:
BLOCKING = {
    "INVALID_EMAIL_FORMAT", "HIGH_INVALID_RATE", "PII_ENV_RISK",
    "NOT_LOWERCASE", "WHITESPACE_NOT_TRIMMED",
}

if not all_findings:
    print("No email columns found matching criteria. Run complete.")
else:
    fdf = findings_df

    # Section 1 -- Blocking (certification blockers)
    block_df = fdf.filter(F.col("classification").isin(BLOCKING)) \
                  .orderBy(F.col("affected_pct").desc())
    n_block = block_df.count()
    print("=" * 70)
    print(f"  SECTION 1 -- BLOCKING FINDINGS: {n_block}")
    print("=" * 70)
    if n_block:
        display(block_df.select(
            "table_name","column_name","semantic_role","pii_risk",
            "classification","rule_ref","affected_count","affected_pct",
            "recommended_action","decided_action","decided_by"
        ))
    else:
        print("  None.")

    # Section 2 -- By classification
    print("\n" + "-" * 70)
    print("SECTION 2 -- Findings by classification")
    print("-" * 70)
    display(
        fdf.groupBy("classification","rule_ref")
           .agg(
               F.count("*").alias("findings"),
               F.countDistinct("table_name").alias("tables"),
               F.countDistinct("column_name").alias("columns"),
               F.sum("affected_count").alias("affected_rows"),
           ).orderBy(F.col("findings").desc())
    )

    # Section 3 -- By table
    print("\n" + "-" * 70)
    print("SECTION 3 -- Findings by table")
    print("-" * 70)
    display(
        fdf.groupBy("table_name")
           .agg(
               F.count("*").alias("total_findings"),
               F.sum(F.when(F.col("classification").isin(BLOCKING), 1).otherwise(0)).alias("blocking"),
               F.countDistinct("column_name").alias("email_columns"),
               F.first("pii_risk").alias("pii_risk"),
           ).orderBy(F.col("blocking").desc(), F.col("total_findings").desc())
    )

    # Section 4 -- PII / privacy
    priv_df = fdf.filter(F.col("classification").isin("PII_ENV_RISK", "PLAIN_EMAIL_IN_LOG_RISK"))
    n_priv  = priv_df.count()
    print("\n" + "-" * 70)
    print(f"SECTION 4 -- PII / privacy exposure ({n_priv})")
    print("  Email is PII -- verify masking in non-production, no plain email in logs")
    print("-" * 70)
    if n_priv:
        display(priv_df.select(
            "table_name","column_name","pii_risk",
            "classification","recommended_action","decided_action","decided_by"
        ).orderBy("classification","table_name","column_name"))
    else:
        print("  None.")

    # Section 5 -- Role accounts and duplicates
    role_df = fdf.filter(F.col("classification").isin(
        "ROLE_ACCOUNT_PRESENT","DUPLICATE_EMAIL_IN_COLUMN"
    ))
    n_role = role_df.count()
    print("\n" + "-" * 70)
    print(f"SECTION 5 -- Role accounts and duplicate risks ({n_role})")
    print("  Shared inboxes must not be used as unique person keys without product owner approval")
    print("-" * 70)
    if n_role:
        display(role_df.select(
            "table_name","column_name","semantic_role","is_likely_unique_key",
            "classification","affected_count","affected_pct",
            "recommended_action","decided_action","decided_by"
        ).orderBy("classification","table_name","column_name"))
    else:
        print("  None.")

    # Section 6 -- Engineer work queue
    work_df = fdf.filter(F.col("decided_action").isNotNull())
    n_work  = work_df.count()
    print("\n" + "-" * 70)
    print(f"SECTION 6 -- Engineer work queue ({n_work} decided)")
    print("-" * 70)
    if n_work:
        display(work_df.select(
            "table_name","column_name","classification",
            "decided_action","decided_by","standardization_rule"
        ).orderBy("table_name","column_name"))
    else:
        print("  No decisions recorded yet.")
        print(f"  Query: SELECT * FROM {CFG['out_catalog']}.{CFG['out_schema']}.{CFG['out_table']}")
        print(f"  WHERE run_id = '{RUN_ID}' AND decided_action IS NULL")

    print("\n" + "=" * 70)
    print(f"  Run: {RUN_ID}")
    print(f"  Scope: {cat}.{sch}  |  Layer: {CFG['layer']}")
    print(f"  Email columns confirmed: {total_confirmed}")
    print(f"  Findings: {len(all_findings):,}  |  Blocking: {n_block}  |  PII risk: {n_priv}  |  Role/dup: {n_role}")
    print("=" * 70)
    print("\nDetection-only. No source data was modified.")
    print("Every finding requires data steward review before any action.")
